In [37]:

import pandas 
import geopandas
import os
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from config import *
from misc_utilities import *

In [38]:
selected_cols = ['X', 'Y', 'Z', 'red', 'green', 'blue','classification']
data_info_df = pandas.read_csv(data_info_file)
filename_paths = {os.path.split(x)[-1]:x for x in data_info_df['filename']}
for las_file in las_vect_dict:
    vect_file = las_vect_dict[las_file]
    shapefile_path = os.path.join(vector_dir,vect_file)
    las_file_path = filename_paths[las_file]
    las_file_name = os.path.splitext(las_file)[0]
    dtm_file = las_file_name+"_dtm.las"
    dtm_file_path = os.path.join(dtm_dir,dtm_file)
    gdf = geopandas.read_file(shapefile_path)
    for row1 in gdf.iterrows():
        id = row1[1]['id']
        geometry = row1[1].geometry
        dtm_record,header = subset_with_geom(dtm_file_path,geometry)
        df = pandas.DataFrame(record.array)
        df_selected = df[selected_cols]
        df_selected.to_csv(os.path.join(debug_dir,f"{las_file_name}_id_{id}.csv"),index=False)

In [39]:
dtm_file_path

'/mnt/nvme1n1/Bipin/Scripts/geo_ai/data/input/dtm/64334_2H_(REFLIGHT)_POINT_CLOUD_dtm.las'

In [40]:
df['classification'].max()

np.uint8(2)

In [41]:
ground_df = df.query("classification==2")
non_ground_df = df.query("classification!=2")

In [42]:
non_ground_df

,X,Y,Z,intensity,bit_fields,classification_flags,classification,user_data,scan_angle,point_source_id,gps_time,red,green,blue
4557,16465,35324,12685,0,17,64,1,0,0,1,0.0,28013,22359,16962
5159,16471,35314,12688,0,17,64,1,0,0,1,0.0,31097,25443,20046
5464,16457,35309,12688,0,17,64,1,0,0,1,0.0,31097,25186,19532
5765,16448,35289,12676,0,17,64,1,0,0,1,0.0,35209,28784,22616
6001,16465,35293,12680,0,17,64,1,0,0,1,0.0,33410,27242,21074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64893,16845,36292,12739,0,17,64,1,0,0,1,0.0,36237,30326,24415
64895,16773,36260,12749,0,17,64,1,0,0,1,0.0,35209,29041,22873
64978,16824,36384,12703,0,17,64,1,0,0,1,0.0,23901,18504,13364
65203,16830,36370,12713,0,17,64,1,0,0,1,0.0,30583,24929,19532


In [43]:
ground_df

,X,Y,Z,intensity,bit_fields,classification_flags,classification,user_data,scan_angle,point_source_id,gps_time,red,green,blue
0,16520,34333,12629,0,17,64,2,0,0,1,0.0,53456,52428,50886
1,16565,34235,12627,0,17,64,2,0,0,1,0.0,23901,19789,16448
2,16514,34212,12626,0,17,64,2,0,0,1,0.0,25957,22616,18504
3,16540,34284,12619,0,17,64,2,0,0,1,0.0,30840,26728,23130
4,16479,34333,12616,0,17,64,2,0,0,1,0.0,35980,33153,29041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72897,15610,37472,12585,0,17,64,2,0,0,1,0.0,40606,34438,27756
72898,15633,37420,12586,0,17,64,2,0,0,1,0.0,40606,34695,28270
72899,15621,37443,12587,0,17,64,2,0,0,1,0.0,39321,32896,26471
72900,15630,37463,12588,0,17,64,2,0,0,1,0.0,41377,35209,28784


In [44]:
df_nan = df[['X',"Y","Z"]]
df_nan.loc[df['classification'] != 2, 'Z'] = np.nan    
imputer = SimpleImputer()
imputed_df = imputer.fit_transform(np.array(df_nan))

In [45]:
imputed_df

array([[16520., 34333., 12629.],
       [16565., 34235., 12627.],
       [16514., 34212., 12626.],
       ...,
       [15621., 37443., 12587.],
       [15630., 37463., 12588.],
       [15653., 37443., 12590.]], shape=(72902, 3))

In [ ]:
knn = NearestNeighbors(n_neighbors=4)
ground_points = np.array(ground_df[["X","Y","Z"]]
knn.fit(ground_points)
distances,indices = knn.kneighbors(np.array(non_ground_df[["X","Y","Z"]]))

In [51]:
indices

array([[ 3953,  4153,  7158,  5088],
       [ 4217,  3540,  3953,  7158],
       [ 4153,  4038,  3824,  4217],
       ...,
       [54636, 56959, 56951, 56735],
       [56047, 58273, 56959, 55434],
       [56047, 56959, 58018, 58273]], shape=(10780, 4))

In [52]:
distances

array([[ 7.87400787, 13.        , 16.76305461, 17.94435844],
       [15.68438714, 15.93737745, 16.88194302, 20.49390153],
       [14.45683229, 17.60681686, 21.44761059, 21.61018278],
       ...,
       [10.19803903, 10.48808848, 10.81665383, 16.82260384],
       [16.88194302, 17.52141547, 20.24845673, 20.63976744],
       [13.37908816, 14.69693846, 17.4642492 , 20.61552813]],
      shape=(10780, 4))